In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [11]:
# =========================
# 1. Imports
# =========================
import os
import re
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset, DataLoader

# =========================
# 2. LOAD DATASET
# =========================
base_path = "/kaggle/input/datasets/nikunjnawal009/bilstm-devignx26"

df = None
for root, dirs, files in os.walk(base_path):
    for f in files:
        path = os.path.join(root, f)
        print("Trying:", path)

        try:
            if f.endswith(".csv"):
                df = pd.read_csv(path)
                print("✅ Loaded:", path)
                break
        except:
            continue
    if df is not None:
        break

if df is None:
    raise Exception("❌ Dataset not found")

print("\n📊 Shape:", df.shape)
print("📌 Columns:", df.columns)

# =========================
# 3. PREPROCESS
# =========================
df = df[['code', 'label']].dropna()
df.columns = ['text', 'label']
df['label'] = df['label'].astype(int)

# =========================
# 4. TRAIN-TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    df["text"].astype(str).tolist(),
    df["label"].values,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

# =========================
# 5. TOKENIZATION
# =========================
def tokenize(text):
    text = text.lower()
    text = re.sub(r'([(){}\[\];,<>!=&|^~*/%+-])', r' \1 ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.split()

counter = Counter()
for t in X_train:
    counter.update(tokenize(t))

vocab = {"<PAD>": 0, "<UNK>": 1}
for word, freq in counter.most_common(20000):
    if freq >= 2:
        vocab[word] = len(vocab)

def encode(text):
    return [vocab.get(w, 1) for w in tokenize(text)]

MAX_LEN = 300

def pad(seq):
    return seq[:MAX_LEN] + [0]*(MAX_LEN - len(seq))

X_train = [pad(encode(t)) for t in X_train]
X_test  = [pad(encode(t)) for t in X_test]

# =========================
# 6. DATASET CLASS
# =========================
class DevignDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self): return len(self.y)

    def __getitem__(self, i):
        return self.X[i], self.y[i]

train_loader = DataLoader(DevignDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader  = DataLoader(DevignDataset(X_test, y_test), batch_size=32)

# =========================
# 7. BiLSTM MODEL
# =========================
class BiLSTMModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, 128, padding_idx=0)

        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=256,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(512, 1)

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        out = torch.cat((hidden[-2], hidden[-1]), dim=1)
        return self.fc(self.dropout(out)).squeeze()

# =========================
# 8. SETUP
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BiLSTMModel(len(vocab)).to(device)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

pos_weight = torch.tensor(class_weights[1]).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# =========================
# 9. TRAINING
# =========================
EPOCHS = 8

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)

        optimizer.zero_grad()
        loss = criterion(model(Xb), yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

# =========================
# 10. EVALUATION
# =========================
model.eval()
preds, true = [], []

with torch.no_grad():
    for Xb, yb in test_loader:
        out = torch.sigmoid(model(Xb.to(device)))
        preds.extend((out > 0.5).cpu().numpy())
        true.extend(yb.numpy())

acc = accuracy_score(true, preds)
report = classification_report(true, preds, output_dict=True)

print("\n✅ Accuracy:", acc)
print("\n📊 Classification Report:\n", classification_report(true, preds))

# =========================
# 11. FIXED METRIC EXTRACTION
# =========================
label_key = None
for key in report.keys():
    if key.startswith("1"):
        label_key = key

results = {
    "Model": "BiLSTM",
    "Dataset": "Devign",
    "Accuracy": acc,
    "Precision_vuln": report[label_key]['precision'],
    "Recall_vuln": report[label_key]['recall'],
    "F1_vuln": report[label_key]['f1-score'],
    "Macro_F1": report['macro avg']['f1-score']
}

# =========================
# 12. SAVE RESULTS
# =========================
pd.DataFrame([results]).to_csv("/kaggle/working/bilstm_devign_results.csv", index=False)

pd.DataFrame({
    "true_label": true,
    "predicted_label": preds
}).to_csv("/kaggle/working/bilstm_devign_predictions.csv", index=False)

torch.save(model.state_dict(), "/kaggle/working/bilstm_devign_model.pth")

print("\n✅ All files saved successfully in /kaggle/working/")

Trying: /kaggle/input/datasets/nikunjnawal009/bilstm-devignx26/Devignx_validation.csv
✅ Loaded: /kaggle/input/datasets/nikunjnawal009/bilstm-devignx26/Devignx_validation.csv

📊 Shape: (2732, 2)
📌 Columns: Index(['code', 'label'], dtype='object')
Epoch 1, Loss: 0.7382
Epoch 2, Loss: 0.7367
Epoch 3, Loss: 0.7358
Epoch 4, Loss: 0.7335
Epoch 5, Loss: 0.7320
Epoch 6, Loss: 0.7281
Epoch 7, Loss: 0.7244
Epoch 8, Loss: 0.7210

✅ Accuracy: 0.5648994515539305

📊 Classification Report:
               precision    recall  f1-score   support

         0.0       0.58      0.83      0.68       309
         1.0       0.50      0.22      0.31       238

    accuracy                           0.56       547
   macro avg       0.54      0.53      0.50       547
weighted avg       0.55      0.56      0.52       547


✅ All files saved successfully in /kaggle/working/
